# Detection Comparison: Classical vs DETR (JaispDetector)

Compare source detection quality between:
- **Classical**: `detect_sources` peak-finding on a g+r+i co-add (no model)
- **DETR**: `JaispDetector` on frozen MAE v6 encoder features (all 10 bands jointly)

Metrics:
- Number of detections vs SNR threshold
- Positional scatter (cross-match radius)
- Star/galaxy separation (concentration index vs model class logit)
- Faint-end completeness

In [1]:
import sys
from pathlib import Path

MODELS = Path('..').resolve()   # models/
for p in [MODELS, MODELS / 'detection', MODELS / 'photometry', MODELS / 'astrometry2']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial import cKDTree

from jaisp_foundation_v6 import JAISPFoundationV6, ALL_BANDS, RUBIN_BANDS
from detection.detector import JaispDetector, JAISPEncoderWrapper
from detection.dataset  import _pseudo_labels, _concentration_index
from astrometry2.source_matching import detect_sources, build_detection_image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

ModuleNotFoundError: No module named 'jaisp_foundation_v6'

## Load tile + models

In [ ]:
RUBIN_DIR  = Path('../../data/rubin_tiles_ecdfs')
ENCODER_CKPT  = Path('../checkpoints/jaisp_v6_phaseB2/checkpoint_best.pt')
DETECTOR_CKPT = Path('../checkpoints/detector_v1.pt')

# Pick a tile (change index to explore different tiles)
TILE_IDX = 0
tile_path = sorted(RUBIN_DIR.glob('tile_x*_y*.npz'))[TILE_IDX]
print(f'Tile: {tile_path.name}')

# Load raw tile
data    = np.load(tile_path, allow_pickle=True)
img_np  = np.nan_to_num(data['img'].astype(np.float32),  nan=0.0)   # [6, H, W]
var_np  = np.nan_to_num(data['var'].astype(np.float32),  nan=0.0)
rms_np  = np.sqrt(np.clip(var_np, 0, None))
C, H, W = img_np.shape
print(f'Tile shape: {img_np.shape}  ({H}×{W} px)')

In [ ]:
# Load MAE encoder + detector
foundation = JAISPFoundationV6(band_names=ALL_BANDS)
ckpt = torch.load(ENCODER_CKPT, map_location='cpu', weights_only=False)
foundation.load_state_dict(ckpt['model'])
encoder = JAISPEncoderWrapper(foundation.encoder, freeze=True).to(DEVICE)
encoder.eval()

detector = JaispDetector.load(str(DETECTOR_CKPT), encoder=encoder, device=DEVICE)
detector.eval()
print('Models loaded.')

## Run both detectors

In [ ]:
# ── Classical detection ───────────────────────────────────────────────────────
det_img = build_detection_image(img_np, ['rubin_g', 'rubin_r', 'rubin_i'])

classical_results = {}
for nsig in [3, 5, 8, 10]:
    xs, ys = detect_sources(det_img, nsig=nsig, max_sources=2000)
    conc   = _concentration_index(img_np[2], xs, ys) if len(xs) else np.array([])
    classical_results[nsig] = dict(xs=xs, ys=ys, conc=conc)

# Use nsig=5 as the primary classical result
cl = classical_results[5]
cl_xy  = np.stack([cl['xs'], cl['ys']], axis=1) if len(cl['xs']) else np.zeros((0,2))
cl_cls = (cl['conc'] < 0.5).astype(int)   # 0=star, 1=galaxy
print(f'Classical (nsig=5): {len(cl_xy)} sources  '
      f'({(cl_cls==0).sum()} stars, {(cl_cls==1).sum()} galaxies)')